# L5: Dirty data and missing-value flags

Profile a deliberately dirty synthetic extract. Preserve raw records, separate quarantine, and create indicators before imputation.

**Environment:** See ../requirements.txt and the labs index. Examples use bundled inputs; no internet is required after dependency installation. Synthetic data is for teaching only.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
ROOT = Path.cwd() if (Path.cwd() / "datasets").exists() else Path.cwd().parent
assert (ROOT / "datasets").exists(), "Run from the repository root or labs directory; in Colab upload/clone the repository first."
DATA = ROOT / "datasets"


## Profile and identify issues

In [ ]:
raw = pd.read_csv(DATA / "dirty_sales.csv")
print(raw.isna().sum())
print("Exact duplicates:",raw.duplicated().sum())
assert raw.Quantity.isna().sum() == 1
assert raw.duplicated().sum() == 1

## Clean without erasing the audit trail

In [ ]:
df = raw.drop_duplicates().copy()
df["Region"] = df.Region.str.strip().str.title()
df["OrderDate"] = pd.to_datetime(df.OrderDate,errors="coerce")
invalid = (df.Quantity < 0) | df.OrderDate.isna()
quarantine = df.loc[invalid].copy()
clean = df.loc[~invalid].copy()
clean["QuantityWasMissing"] = clean.Quantity.isna()
median = clean.Quantity.median()
clean["Quantity"] = clean.Quantity.fillna(median)
clean["GrossRevenue"] = clean.Quantity * clean.UnitPrice
clean["NetRevenue"] = clean.GrossRevenue * (1-clean.Discount)
clean["TotalCost"] = clean.Quantity * clean.UnitCost
clean["Profit"] = clean.NetRevenue-clean.TotalCost
assert clean.QuantityWasMissing.sum() == 1
assert clean.Quantity.eq(100).any()
print("Accepted",len(clean),"quarantined",len(quarantine),"median baseline",median)
print(clean.loc[clean.QuantityWasMissing,["OrderID","Quantity","QuantityWasMissing"]])

## Missingness interpretation

## Student task
Explain why the indicator survives filling. Compare results without the imputed row. Do not claim MAR or MNAR is established by observed patterns. The median is a descriptive teaching baseline; prediction uses train-fitted imputers and inference needs uncertainty-aware methods.

## Reference solution
Try the task before reading this cell.

In [ ]:
print("Revenue sensitivity:",clean.NetRevenue.sum(),clean.loc[~clean.QuantityWasMissing,"NetRevenue"].sum())

## Interpretation and limitations
Record the dataset grain, units, assumptions, and a limitation in your submission. Explain one result rather than only submitting screenshots.